In [ ]:
from mltau.tools.evaluation import evaluate_daughter_model as ed
from mltau.tools.evaluation import decode_HPS as dh

In [ ]:
import awkward as ak
import os
import mplhep
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main_ParTauDETR")

In [ ]:
HPS_pred_path = "/home/laurits/20260726_HPS/z_test.parquet"
ParTauDETR_path = "/home/laurits/tmp/ParTauDETR_pred.parquet"

In [ ]:
hps_data = ak.from_parquet(HPS_pred_path)
sts_data = ak.from_parquet(ParTauDETR_path)

In [ ]:
charged_pdg = [321, 211]
neutral_pdg = [311, 221, 111, 130]

In [ ]:
def count_ch_neutral(pdg):
    charged = ak.sum(
        (abs(pdg) == 321) | (abs(pdg) == 211),
        axis=1
    )
    
    neutral = ak.sum(
        (abs(pdg) == 311) |
        (abs(pdg) == 221) |
        (abs(pdg) == 111) |
        (abs(pdg) == 130),
        axis=1
    )
    return charged, neutral

In [ ]:
def get_decay_mode(n_charged, n_neutral):
    decay_mode = 5 * (n_charged - 1) + n_neutral
    return decay_mode

In [ ]:
n_pred_charged, n_pred_neutral = count_ch_neutral(sts_data.pred_pdg)
n_true_charged, n_true_neutral = count_ch_neutral(sts_data.true_pdg)
pred_dm = get_decay_mode(n_pred_charged, n_pred_neutral)
true_dm = get_decay_mode(n_true_charged, n_true_neutral)

In [ ]:
sts_pred_dm0_mask = pred_dm == 0
sts_true_dm0_mask = true_dm == 0

from mltau.tools import general as g
pred_energy = g.reinitialize_p4(sts_data.pred_p4).energy[sts_pred_dm0_mask]
true_energy = g.reinitialize_p4(sts_data.true_p4).energy[sts_true_dm0_mask]
pred_sts_x = ed.calculate_x_dm0(E_pi=pred_energy)
true_sts_x = ed.calculate_x_dm0(E_pi=true_energy)

In [ ]:
pred_hps_x, true_hps_x = ed.hps_x_dm0(hps_data)
predictions = {
    "HPS": pred_hps_x,
    "STS": ak.flatten(pred_sts_x),
}

In [ ]:
ed.plot_performance(predictions, true_hps_x)


In [ ]:
truth = hps_data.gen_jet_tau_decaymode
predicted = hps_data.tau_decaymode
categories = np.unique(hps_data.gen_jet_tau_decaymode)


In [ ]:
dh.evaluate_hps_decaymode_classification(truth, predicted, categories)

In [ ]:
from mltau.tools.evaluation import kinematics as k

In [ ]:
hps_ke = k.KinematicsEvaluator(
    predicted_p4=hps_data.tau_p4s,
    true_p4=hps_data.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="HPS",
    sample_name='z'
)
    


In [ ]:
kme = k.KinematicsMultiEvaluator(output_dir="/home/laurits/HPS_kin_eval", cfg=cfg, sample="z")
kme.combine_results([hps_ke])